In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
df = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')

In [3]:
# On drop les colonnes en lien avec les notes/temps de réponse des maths sauf math_q1_total_timing qui n'est pas vide dans le jeu test

math_cols = [col for col in df.columns if col.startswith("math_q") and col != "math_q1_total_timing"]
df = df.drop(columns=math_cols)

In [4]:
# X on convertit les objects en catégories
obj_cols = df.select_dtypes(include="object").columns

for col in obj_cols:
    df[col] = df[col].astype("category")

In [5]:
X_train = df
X_train = X_train.drop(columns=['Unnamed: 0'])
y_train = y_train.drop(columns=['Unnamed: 0'])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_train,                  # tes features
    y_train,                  # ta cible
    test_size=0.2,      # 20% en test
    random_state=42,    # pour un split reproductible
    shuffle=True        # mélanger avant de couper, très recommandé
)

In [7]:
xgb = XGBRegressor(
    n_estimators=600,       # nombre d’arbres
    learning_rate=0.05,    # pas d’apprentissage
    tree_method="hist",    # obligatoire pour enable_categorical
    enable_categorical=True,
    max_depth=6,           # profondeur des arbres
    subsample=0.8,         # échantillonnage aléatoire
    colsample_bytree=0.8,  # échantillonnage des features
    objective='reg:squarederror',
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R² :", r2)
print("RMSE :", rmse)

R² : 0.773587703704834
RMSE : 58.07046753056798


In [15]:
X_jeu_test = pd.read_csv('X_test.csv')

In [16]:
# On drop les colonnes en lien avec les notes/temps de réponse des maths sauf math_q1_total_timing qui n'est pas vide dans le jeu test

math_cols = [col for col in X_jeu_test.columns if col.startswith("math_q") and col != "math_q1_total_timing"]
X_jeu_test = X_jeu_test.drop(columns=math_cols)

In [17]:
# X on convertit les objects en catégories
obj_cols = X_jeu_test.select_dtypes(include="object").columns

for col in obj_cols:
    X_jeu_test[col] = X_jeu_test[col].astype("category")

In [18]:
id_col = X_jeu_test['Unnamed: 0'].copy()
X_jeu_test = X_jeu_test.drop(columns=['Unnamed: 0'])

In [19]:
y_pred = xgb.predict(X_jeu_test)

In [28]:
df_pred = pd.DataFrame({
    "ID": id_col,
    "MathScore": y_pred
})


In [29]:
df_pred.to_csv("predictions.csv", index=False)

In [30]:
df_pred

,ID,MathScore
0,412660,130.115616
1,554658,93.452782
2,937138,-0.079194
3,752986,216.742889
4,1084508,123.007812
...,...,...
586039,1757496,210.229385
586040,1414197,5.748414
586041,821972,235.881287
586042,25376,73.558937
